# Stats LIFF: 分析サンプルノートブック

BigQueryに蓄積された学習ログから、合格戦略を立てるための代表的な分析を行うサンプル。

前提:
- `gcloud auth application-default login` 済
- `pip install google-cloud-bigquery pandas scipy matplotlib`

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery

PROJECT = os.environ.get('GCP_PROJECT', 'your-project-id')
USER_HASH = os.environ.get('USER_HASH', 'your-user-hash')

bq = bigquery.Client(project=PROJECT)

## 1. 直近30日の分野別パフォーマンス

In [ ]:
df = bq.query(f'''
  SELECT * FROM `{PROJECT}.stats_mart.v_topic_perf_30d`
  WHERE user_id_hash = @uid
  ORDER BY cv_time DESC
''', job_config=bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter('uid', 'STRING', USER_HASH)]
)).to_dataframe()
df.head(10)

## 2. 不安定ゾーンの可視化（accuracy × CV）

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['accuracy'], df['cv_time'], s=df['n_attempt']*5, alpha=0.6)
ax.axhline(0.5, ls='--', color='red', label='CV=0.5 (不安定境界)')
ax.axvline(0.7, ls='--', color='green', label='正解率70%')
ax.set_xlabel('正解率')
ax.set_ylabel('解答時間の変動係数 CV')
ax.set_title('右下 = 正解率は高いが解答時間が不安定 = 未定着の疑い')
ax.legend()
plt.tight_layout()
plt.show()

## 3. 解答時間の分布（対数変換の必要性チェック）

[KNOWN_LIMITATIONS §2.1](../../docs/KNOWN_LIMITATIONS.md) で指摘した、
解答時間が対数正規に近いかをヒストグラムで確認する。

In [ ]:
import numpy as np

raw = bq.query(f'''
  SELECT time_sec FROM `{PROJECT}.stats_mart.fact_answer`
  WHERE user_id_hash = @uid AND time_outlier_flag = FALSE
''', job_config=bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter('uid', 'STRING', USER_HASH)]
)).to_dataframe()['time_sec']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw, bins=30); axes[0].set_title('time_sec (生)')
axes[1].hist(np.log(raw[raw > 0]), bins=30); axes[1].set_title('log(time_sec)')
plt.tight_layout(); plt.show()